# Particle tracking and the water table

## What this notebook covers

Build a transient MODFLOW 6 **groundwater flow (GWF)** model whose upper layers start dry and wet up over time, then trace particles through it with a **particle-tracking (PRT)** model that reuses the flow model's saved heads and budgets. The catch: particles are released into the **dry** top layer, and how PRT handles that depends on the flow model.

You run it **twice**:

1. **Without Newton** — MODFLOW makes the dry cells inactive, so PRT uses `drape` to drop each particle onto the water table at release.
2. **With Newton** — the dry cells stay active, so PRT tracks particles through them and `dry_tracking_method` decides what they do.

By the end you can feed flow output to a PRT model, plot pathlines in map view and 3D, and reason about PRT's behavior in dry conditions.

Reference docs: the [MODFLOW 6 input/output guide](https://modflow6.readthedocs.io/en/latest/mf6io.html) and the [FloPy documentation](https://flopy.readthedocs.io/).

#### Define the units

MODFLOW 6 has no built-in unit system; it only requires that every input share a consistent length and time unit. Use meters and days throughout this notebook, and set `time_unit` and `length_unit` so the discretization packages record them.

In [ ]:
time_unit = "days"
length_unit = "meters"

#### Set up the temporal discretization

Divide the simulation into 3 **stress periods** — blocks of time over which the boundary conditions are held constant: a steady-state period, a transient period, and a final steady-state period. The injection well is active in the transient period. Each row of `tdis_pd` is one period, given as a tuple:

```python
# (period length, number of time steps, time-step multiplier)
(300, 30, 1.1)
```

so the middle period runs 300 days, split into 30 time steps whose length grows by a factor of 1.1 each.

In [ ]:
nper = 3
tdis_pd = [
    (1, 1, 1.0),
    (300, 30, 1.1),
    (1000, 1, 1.0),
]

#### Set up the spatial discretization

Build a grid of 3 layers, 20 rows, and 20 columns. Each layer is 10 m thick below a top surface at 1600 m, so compute each layer bottom (`bot`) by subtracting the cumulative thickness from `top`. Cell IDs throughout are zero-based `(layer, row, column)` tuples counting from 0.

In [ ]:
import numpy as np

nlay, nrow, ncol = 3, 20, 20
Lx, Ly = 100.0, 100.0
delr, delc = Lx / ncol, Ly / nrow
top = 1600.0
thickness = 10.0
bot = np.zeros((nlay, nrow, ncol))
for k in range(nlay):
    bot[k] = top - (k + 1) * thickness

#### Set up the workspaces

Create a base workspace for the whole example and a `gwf` subfolder for the flow model. `pathlib.Path` builds OS-independent paths, and `mkdir(parents=True)` creates the folders. Keeping the flow and particle-tracking models in separate folders matters later, when PRT reads the flow model's output files by relative path.

In [ ]:
from pathlib import Path

example_name = "prt-wt"
base_ws = Path("models") / example_name
gwf_name = f"{example_name}-gwf"
gwf_ws = base_ws / "gwf"
gwf_ws.mkdir(exist_ok=True, parents=True)

## Phase 1 — Build the flow model (no Newton)

Everything up to the dry-cell check builds and runs the GWF model. Its saved heads and cell budgets become the input to the PRT model in Phase 2.

Create the simulation, temporal discretization, model, spatial discretization, and initial conditions in one cell with `flopy.mf6.MFSimulation()`, `ModflowTdis()`, `ModflowGwf()`, `ModflowGwfdis()`, and `ModflowGwfic()`. Set the starting head (`strt`) to the top of the model. This first pass uses the standard formulation, so the dry upper cells drop out of the solution (go inactive); Phase 3 turns on Newton to keep them active.

In [ ]:
import flopy

gwf_sim = flopy.mf6.MFSimulation(
    sim_name=gwf_name,
    exe_name="mf6",
    version="mf6",
    sim_ws=gwf_ws,
)
gwf_tdis = flopy.mf6.ModflowTdis(
    gwf_sim,
    time_units=time_unit,
    nper=nper,
    perioddata=tdis_pd,
)
gwf = flopy.mf6.ModflowGwf(
    gwf_sim,
    modelname=gwf_name,
    save_flows=True,
)
gwf_dis = flopy.mf6.ModflowGwfdis(
    gwf,
    length_units=length_unit,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)
gwf_ic = flopy.mf6.ModflowGwfic(gwf, strt=top)

#### Node Property Flow (NPF) package

Set the aquifer's hydraulic conductivity with `flopy.mf6.ModflowGwfnpf()`. `icelltype=1` makes every cell **convertible** — able to switch between confined and unconfined as it wets and dries — which is what lets the upper layers go dry. Horizontal conductivity `k=0.5` exceeds vertical `k33=0.1`, so water moves more easily laterally than vertically. Save specific discharge and saturation for later use.

In [ ]:
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=1,
    k=0.5,
    k33=0.1,
    save_specific_discharge=True,
    save_saturation=True,
    save_flows=True,
)

#### Storage (STO) package

Add aquifer storage with `flopy.mf6.ModflowGwfsto()` so the transient period can fill and drain. Match `iconvert=1` to the NPF cell type, set specific storage `ss` and specific yield `sy`, and mark which periods are steady-state versus transient using dicts keyed by the zero-based period number — periods 0 and 2 steady, period 1 transient.

In [ ]:
sto = flopy.mf6.ModflowGwfsto(
    gwf,
    iconvert=1,
    ss=0.0001,
    sy=0.1,
    steady_state={0: True, 2: True},
    transient={1: True},
    save_flows=True,
)

#### Constant-head (CHD) boundaries

Pin the head at two cells with `flopy.mf6.ModflowGwfchd()` to drive a gentle gradient across the grid, from 1587 m in the upper-left to 1582 m in the lower-right. `CHD` is a list-based stress package: its `stress_period_data` is a dict keyed by the zero-based stress period, and each entry is a tuple of

```python
# (layer, row, column, head)
(1, 0, 0, 1587.0)
```

The same two cells apply in every period, so the dict comprehension repeats them for periods 0 through 2.

In [ ]:
chd0_cellid = (1, 0, 0)
chd1_cellid = (1, nrow - 1, ncol - 1)
chd = flopy.mf6.ModflowGwfchd(
    gwf,
    stress_period_data={
        i: [(*chd0_cellid, 1587.0), (*chd1_cellid, 1582.0)] for i in range(nper)
    },
    save_flows=True,
)

#### Well (WEL) package

Add an injection well in the middle layer of the upper-left quadrant with `flopy.mf6.ModflowGwfwel()`. Each `WEL` entry is a tuple of

```python
# (layer, row, column, rate)
(1, 5, 5, 100.0)
```

A positive rate injects water, here at 100 m³/d. The dict comprehension over `range(1, nper)` activates the well from the second stress period onward.

In [ ]:
wel_cellid = (1, nrow // 4, ncol // 4)
well = flopy.mf6.ModflowGwfwel(
    gwf,
    stress_period_data={i: [(*wel_cellid, 100.0)] for i in range(1, nper)},
    save_flows=True,
)

#### Output Control (OC) package

Tell MODFLOW which results to write and where with `flopy.mf6.ModflowGwfoc()`. Save heads to a `.hds` file and cell-by-cell flows to a `.cbb` file for every time step. The PRT model reads exactly these two files, so both must be saved.

In [ ]:
oc = flopy.mf6.ModflowGwfoc(
    gwf,
    budget_filerecord=f"{gwf_name}.cbb",
    head_filerecord=f"{gwf_name}.hds",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)

#### Iterative Model Solution (IMS)

Configure the solver with `flopy.mf6.ModflowIms()`. For the standard formulation, `complexity="MODERATE"` and the head-change closure tolerances below are enough. (Phase 3 swaps in a more robust solver for the harder-to-converge Newton run.)

In [ ]:
ims = flopy.mf6.ModflowIms(
    gwf_sim,
    complexity="MODERATE",
    outer_dvclose=1e-5,
    inner_dvclose=1e-6,
    pname=gwf_name,
)

#### Write and run the flow model

Write the MODFLOW 6 input files with `gwf_sim.write_simulation()` and run them with `.run_simulation()`. The `assert` stops the notebook if MODFLOW does not finish normally.

In [ ]:
gwf_sim.write_simulation(silent=True)
success, buff = gwf_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

#### Load and plot the heads

Load the head output with `gwf.output.head()`, then pull the array at the end of each stress period with `.get_data(kstpkper=...)`, where `kstpkper` is a zero-based `(time step, stress period)` pair. Plot the middle-layer head at the end of each period.

The **water table** is the top of the saturated zone. During the initial steady-state period the constant-head boundaries drain the system until the top layer sits entirely above the water table and goes dry. Over the transient period the injection well and boundaries raise the water table until the top layer is partially saturated again. This rise and fall is exactly what the PRT model must cope with when it releases particles into the top layer.

In [ ]:
hds = gwf.output.head()
hds_kper0 = hds.get_data(kstpkper=(0, 0))
hds_kper1 = hds.get_data(kstpkper=(0, 1))
hds_kper2 = hds.get_data(kstpkper=(0, 2))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

with flopy.plot.styles.USGSPlot():
    fig, axes = plt.subplots(1, 3, figsize=(11, 5))
    for i, (ax, arr, title) in enumerate(
        [
            (axes[0], hds_kper0[1], "Layer 1 head, end of period 1"),
            (axes[1], hds_kper1[1], "Layer 1 head, end of period 2"),
            (axes[2], hds_kper2[1], "Layer 1 head, end of period 3"),
        ]
    ):
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=10)
        if i == 0:
            ax.legend(
                handles=[
                    Patch(color="red", label="WELL"),
                    Patch(color="blue", label="CHD"),
                ],
            )
        mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=1)
        mm.plot_grid(alpha=0.2)
        pc = mm.plot_array(arr, alpha=0.5, vmin=1582, vmax=1587)
        plt.colorbar(pc, ax=ax, shrink=0.25, label="Head (m)")
        mm.plot_bc("WEL", plotAll=True, kper=1, color="red")
        mm.plot_bc("CHD", plotAll=True, color="steelblue")

    fig.tight_layout()
    plt.show()

**What to look for.** Head is highest near the upper-left constant-head cell (blue) and lowest near the lower-right one, so water moves down the diagonal gradient. Between the panels the head field shifts as the injection well (red) switches on in the transient period and the water table rises. Cells drawn with no color have drained below the plotted range.

#### Confirm the top layer went dry

Count how many top-layer cells have a head below the cell bottom (`bot[0]`, at 1590 m) after the first period — a head below the cell bottom means the cell is dry. This confirms that particles released into the top layer at the start of the simulation begin in dry cells.

In [ ]:
dry_layer0 = (hds_kper0[0] < bot[0]).sum()
print(
    f"Dry cells in top layer after first stress period: {dry_layer0} of {nrow * ncol}"
)

## Phase 2 — Build the particle-tracking model

The flow model is done and its heads and budgets are saved to disk. Now build a separate PRT model that reads those files and traces particles through the flow field the GWF model computed. PRT does not re-solve for flow — it consumes the GWF output.

Start by choosing where particles begin. Release them from cells in the top layer around the cell directly above the injection well. Each offset in the next cell is a zero-based `(dlayer, drow, dcolumn)` shift; here they pick the cell one layer above the well and its four edge neighbors.

In [ ]:
offsets = [(-1, 0, 0), (-1, -1, 0), (-1, 1, 0), (-1, 0, -1), (-1, 0, 1)]
release_cells = [
    (wel_cellid[0] + dk, wel_cellid[1] + di, wel_cellid[2] + dj)
    for dk, di, dj in offsets
]
release_cells

#### Turn release cells into release points

PRT needs the x, y, z coordinates of each particle, not just cell IDs. Rather than compute them by hand, reuse FloPy's MODPATH 7 (**MP7**) release-template utilities and convert the result to PRT's format. `LRCParticleData` fills the **layer-row-column bounding box** of the chosen cells — here a 3×3 block, 9 cells — with one particle each, and `.to_prp()` converts those into PRT release points on the flow model's grid.

In [ ]:
mp7_cell_data = flopy.modpath.CellDataType(
    rowcelldivisions=1, columncelldivisions=1, layercelldivisions=1
)
release_cells_T = np.array(release_cells).T
lrcregions = [
    [
        [
            min(release_cells_T[0]),
            min(release_cells_T[1]),
            min(release_cells_T[2]),
            max(release_cells_T[0]),
            max(release_cells_T[1]),
            max(release_cells_T[2]),
        ]
    ]
]
mp7_particle_data = flopy.modpath.LRCParticleData(
    subdivisiondata=mp7_cell_data,
    lrcregions=lrcregions,
)
mp7_pg = flopy.modpath.ParticleGroupLRCTemplate(
    particledata=mp7_particle_data,
)
release_pts = list(mp7_particle_data.to_prp(gwf.modelgrid))
release_pts

#### Assemble the PRT model

Build the PRT simulation the same way as the flow model, matching its temporal and spatial discretization exactly so the two grids line up. The PRT-specific packages are:

- **MIP** (Model Input Package, `ModflowPrtmip`) — the aquifer `porosity`, which converts cell fluxes to particle velocities.
- **PRP** (Particle Release Point package, `ModflowPrtprp`) — the release points from the previous step. `drape=True` handles a particle released into a dry cell by dropping it down to the first active cell beneath, i.e. onto the water table, instead of terminating it.
- **OC** (`ModflowPrtoc`) — write the particle tracks to `.trk` and `.csv` files.
- **FMI** (Flow Model Interface, `ModflowPrtfmi`) — the link to the flow model. It points at the GWF `.hds` and `.cbb` files (by relative path, hence the separate workspaces) and is how PRT reads the pre-computed flow field.
- **EMS** (`ModflowEms`) — the explicit solver PRT uses to advance particles.

The **drape-to-water-table** behavior is the key idea here: particles start in dry top-layer cells, and `drape` slides each one down to where the water actually is before tracking begins.

In [ ]:
import os

prt_name = f"{example_name}-prt"
prt_ws = base_ws / "prt"
prt_ws.mkdir(exist_ok=True, parents=True)

prt_sim = flopy.mf6.MFSimulation(
    sim_name=prt_name,
    exe_name="mf6",
    version="mf6",
    sim_ws=prt_ws,
)
prt_tdis = flopy.mf6.ModflowTdis(
    prt_sim,
    time_units=time_unit,
    nper=nper,
    perioddata=tdis_pd,
)
prt = flopy.mf6.ModflowPrt(
    prt_sim,
    modelname=prt_name,
    model_nam_file=f"{prt_name}.name",
)
prt_dis = flopy.mf6.ModflowPrtdis(
    prt,
    length_units=length_unit,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)
prt_mip = flopy.mf6.ModflowPrtmip(prt, porosity=0.1)
prt_prp = flopy.mf6.ModflowPrtprp(
    prt,
    nreleasepts=len(release_pts),
    packagedata=release_pts,
    nreleasetimes=1,
    releasetimes=[(0.0,)],
    print_input=True,
    drape=True,
)
prt_oc = flopy.mf6.ModflowPrtoc(
    prt,
    budget_filerecord=[f"{prt_name}.bud"],
    track_filerecord=[f"{prt_name}.trk"],
    trackcsv_filerecord=[f"{prt_name}.csv"],
    saverecord=[("BUDGET", "ALL")],
    ntracktimes=1,
    tracktimes=[(100.0,)],
)
rel_gwf_ws = os.path.relpath(gwf_ws, prt_ws)
prt_fmi = flopy.mf6.ModflowPrtfmi(
    prt,
    packagedata=[
        ("GWFHEAD", f"{rel_gwf_ws}/{gwf_name}.hds"),
        ("GWFBUDGET", f"{rel_gwf_ws}/{gwf_name}.cbb"),
    ],
)
prt_ems = flopy.mf6.ModflowEms(prt_sim, pname="ems", filename=f"{prt_name}.ems")

#### Write and run the PRT model

Write and run the PRT model just like the flow model. Because FMI points at the flow output, this run traces particles through the heads and budgets the flow model already produced.

In [ ]:
prt_sim.write_simulation(silent=False)
success, buff = prt_sim.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

#### Load and plot the pathlines

Read the particle track CSV into a pandas dataframe with `pd.read_csv()`. Draw the pathlines over the top-layer head with `PlotMapView.plot_pathline()`, and color each recorded particle position by its travel time `t` with a scatter plot.

In [ ]:
import pandas as pd

pls_prt = pd.read_csv(prt_ws / f"{prt_name}.csv")

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.set_aspect("equal")
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=0)
    mm.plot_grid(alpha=0.2, lw=0.4)
    mm.plot_array(hds_kper1[1], alpha=0.25, vmin=1582, vmax=1587)
    mm.plot_bc("WEL", plotAll=True, kper=1, color="red")
    mm.plot_bc("CHD", plotAll=True, color="steelblue")
    mm.plot_pathline(pls_prt, layer="all", alpha=0.5, linewidth=0.5)
    pc = ax.scatter(pls_prt.x, pls_prt.y, c=pls_prt.t, s=5)
    cb = plt.colorbar(pc, shrink=0.5, pad=0.1)
    cb.ax.set_xlabel(r"Time (days)")
    ax.set_title("Pathlines, points colored by travel time", fontsize=11)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="CHD"),
        ],
    )

    fig.tight_layout()
    plt.show()

**What to look for.** Each line is one particle's path from its release cell in the top layer toward the constant-head outlet, and the scatter colors show travel time — darker points are earlier, lighter points later — so you can see where particles move quickly versus linger. Because `drape=True`, particles released into the dry top layer first drop onto the water table and then track laterally through the saturated cells; none are stranded in the dry region. The paths respond to the injection well (red), which adds water during the transient period.

#### Plot the pathlines in 3D

Render the same result in 3D with **PyVista**. Export the grid and pathlines to VTK meshes with FloPy's `Vtk` helper, add simple boxes to mark the well and constant-head cells, and color the particle tracks by travel time. The 3D view makes the drape-to-water-table behavior visible: particles drop from the dry top layer down onto the water table and then move laterally.

In [ ]:
import pyvista as pv
from flopy.export.vtk import Vtk


def get_nn(i, j):
    """Convert a structured grid 2D cell index to a node number."""
    return i * ncol + j


def cell_box(cellid):
    """A pv.Box spanning cell (layer, row, col)'s footprint and layer thickness."""
    k, i, j = cellid
    pts = gwf.modelgrid.verts[gwf.modelgrid.iverts[get_nn(i, j)]]
    xs, ys = zip(*pts, strict=False)
    return pv.Box(
        bounds=[min(xs), max(xs), min(ys), max(ys), bot[k].max(), bot[k - 1].max()]
    )


# boxes marking the WELL and the two CHD boundary cells
wel_mesh = cell_box(wel_cellid)
chd_mesh = cell_box(chd0_cellid).merge(cell_box(chd1_cellid))

# create meshes for model results
vtk = Vtk(model=gwf, binary=False)
vtk.add_model(gwf)
vtk.add_pathline_points(pls_prt.to_records(index=False))
gwf_mesh, prt_mesh = vtk.to_pyvista()

# create plot
axes = pv.Axes(show_actor=False, actor_scale=2.0, line_width=5)
p = pv.Plotter(window_size=[700, 700])
p.enable_anti_aliasing()
p.add_mesh(gwf_mesh, opacity=0.1, style="wireframe")
p.add_mesh(wel_mesh, color="red", opacity=0.5)
p.add_mesh(chd_mesh, color="blue", opacity=0.5)
p.add_mesh(
    prt_mesh,
    point_size=8,
    line_width=2.5,
    smooth_shading=True,
    color="blue",
    scalars="t",
    scalar_bar_args={"title": "Time (days)"},
)
p.camera.elevation -= 25
p.show()

#### How PRT decides what to do in dry conditions

`drape` is one branch of two decision trees PRT walks for every particle. At **release** (before tracking starts) PRT checks whether the release cell is active; if it is not and `drape` is on, PRT drapes the particle down to the first active cell — the water table — or terminates it if none is active. During **tracking**, if a particle enters a dry-but-active cell (as happens under Newton) the `dry_tracking_method` option decides whether to stop it, drop it to the water table, or leave it stationary. Phase 1 exercised the release branch; **Phase 3 below turns on Newton and exercises the tracking branch.**

### Release

```mermaid
flowchart LR
    ACTIVE --> |No| DRAPE(DRAPE)
    ACTIVE{Cell active?} --> |Yes| RELEASE[Release in specified cell]:::release
    DRAPE ==> |No| TERMINATE:::terminate
    DRAPE ==> |Yes| ACTIVE_UNDER{Active under?}
    ACTIVE_UNDER --> |Yes| RELEASE_AT_TABLE[Drape to active cell]:::release
    ACTIVE_UNDER --> |No| TERMINATE[Terminate]

    classDef release stroke:#98fb98
    classDef terminate stroke:#f08080
```

### Tracking

```mermaid
flowchart LR
    ACTIVE{Cell active?} --> |No| TERMINATE{Terminate}
    ACTIVE{Cell active?} --> |Yes| PARTICLE_DRY
    PARTICLE_DRY{Particle dry?} --> |Yes| DRY_TRACKING_METHOD(DRY_TRACKING_METHOD)
    DRY_TRACKING_METHOD ==> |STOP| TERMINATE[Terminate]:::terminate
    DRY_TRACKING_METHOD ==> |DROP| CELL_DRY{Cell dry?}
    CELL_DRY --> |Yes| DROP_BOTTOM[Pass to cell bottom]:::track
    CELL_DRY --> |No| DROP_TABLE([Pass to water table])
    DRY_TRACKING_METHOD ==> |STAY| STAY[Stationary]:::track
    DROP_TABLE --> TRACK:::track
    PARTICLE_DRY --> |No| TRACK[Track normally]

    classDef track stroke:#98fb98
    classDef terminate stroke:#f08080
```

## Phase 3 — Second pass: keep dry cells active with Newton

In Phase 1 the dry top-layer cells went **inactive**, so a particle released there needed `drape` to slide down to the water table (the *release* branch above). Turn on the **Newton** formulation and those cells stay **active even when dry** — so particles release and track right inside them, and the *tracking* branch (`dry_tracking_method`) governs what they do.

Rebuild the flow model with two changes: add `newtonoptions="NEWTON UNDER_RELAXATION"` to the model, and swap in the more robust solver Newton needs (DBD under-relaxation, a `BICGSTAB` accelerator). Every other package is exactly as in Phase 1.

In [ ]:
gwf_ws_n = base_ws / "gwf-newton"
gwf_ws_n.mkdir(exist_ok=True, parents=True)
gwf_name_n = f"{example_name}-gwf-nwt"

gwf_sim_n = flopy.mf6.MFSimulation(
    sim_name=gwf_name_n, exe_name="mf6", version="mf6", sim_ws=gwf_ws_n
)
flopy.mf6.ModflowTdis(gwf_sim_n, time_units=time_unit, nper=nper, perioddata=tdis_pd)
gwf_n = flopy.mf6.ModflowGwf(
    gwf_sim_n,
    modelname=gwf_name_n,
    newtonoptions="NEWTON UNDER_RELAXATION",  # keep dry cells active
    save_flows=True,
)
flopy.mf6.ModflowGwfdis(
    gwf_n,
    length_units=length_unit,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)
flopy.mf6.ModflowGwfic(gwf_n, strt=top)
flopy.mf6.ModflowGwfnpf(
    gwf_n,
    icelltype=1,
    k=0.5,
    k33=0.1,
    save_specific_discharge=True,
    save_saturation=True,
    save_flows=True,
)
flopy.mf6.ModflowGwfsto(
    gwf_n,
    iconvert=1,
    ss=0.0001,
    sy=0.1,
    steady_state={0: True, 2: True},
    transient={1: True},
    save_flows=True,
)
flopy.mf6.ModflowGwfchd(
    gwf_n,
    save_flows=True,
    stress_period_data={
        i: [(*chd0_cellid, 1587.0), (*chd1_cellid, 1582.0)] for i in range(nper)
    },
)
flopy.mf6.ModflowGwfwel(
    gwf_n,
    save_flows=True,
    stress_period_data={i: [(*wel_cellid, 100.0)] for i in range(1, nper)},
)
flopy.mf6.ModflowGwfoc(
    gwf_n,
    budget_filerecord=f"{gwf_name_n}.cbb",
    head_filerecord=f"{gwf_name_n}.hds",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)
# Newton is harder to converge, so use a more robust solver configuration
flopy.mf6.ModflowIms(
    gwf_sim_n,
    print_option="SUMMARY",
    outer_dvclose=1e-5,
    outer_maximum=100,
    under_relaxation="DBD",
    under_relaxation_gamma=0.01,
    under_relaxation_theta=0.7,
    under_relaxation_kappa=0.01,
    under_relaxation_momentum=0.0,
    inner_maximum=100,
    inner_dvclose=1e-6,
    rcloserecord=0.1,
    linear_acceleration="BICGSTAB",
    relaxation_factor=0.99,
    number_orthogonalizations=2,
    reordering_method="NONE",
)
gwf_sim_n.write_simulation(silent=True)
success, buff = gwf_sim_n.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

Check the top layer. After the first period its heads are still below the cell bottom (dry) — but with Newton the cells stay **active**, so PRT can release and track particles in them instead of needing `drape`.

In [ ]:
hds_n = gwf_n.output.head()
h0_n = hds_n.get_data(kstpkper=(0, 0))
dry_n = (h0_n[0] < bot[0]).sum()
print(f"Dry top-layer cells after first period: {dry_n} of {nrow * ncol}")
print("With Newton these dry cells remain active, so particles can track through them.")

Build a PRT model against the Newton flow output. Two changes from Phase 2: drop `drape` — the release cells are active now, so it is not needed — and set `dry_tracking_method="drop"`, which passes a particle that enters a dry-but-active cell straight down to the water table. (`"stop"` would terminate it; `"stay"` would freeze it.)

In [ ]:
prt_name_n = f"{example_name}-prt-nwt"
prt_ws_n = base_ws / "prt-newton"
prt_ws_n.mkdir(exist_ok=True, parents=True)

prt_sim_n = flopy.mf6.MFSimulation(
    sim_name=prt_name_n, exe_name="mf6", version="mf6", sim_ws=prt_ws_n
)
flopy.mf6.ModflowTdis(prt_sim_n, time_units=time_unit, nper=nper, perioddata=tdis_pd)
prt_n = flopy.mf6.ModflowPrt(prt_sim_n, modelname=prt_name_n)
flopy.mf6.ModflowPrtdis(
    prt_n,
    length_units=length_unit,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)
flopy.mf6.ModflowPrtmip(prt_n, porosity=0.1)
flopy.mf6.ModflowPrtprp(
    prt_n,
    nreleasepts=len(release_pts),
    packagedata=release_pts,
    nreleasetimes=1,
    releasetimes=[(0.0,)],
    dry_tracking_method="drop",  # cells are active now, so no drape needed
)
flopy.mf6.ModflowPrtoc(
    prt_n,
    budget_filerecord=[f"{prt_name_n}.bud"],
    trackcsv_filerecord=[f"{prt_name_n}.csv"],
    saverecord=[("BUDGET", "ALL")],
)
rel_gwf_ws_n = os.path.relpath(gwf_ws_n, prt_ws_n)
flopy.mf6.ModflowPrtfmi(
    prt_n,
    packagedata=[
        ("GWFHEAD", f"{rel_gwf_ws_n}/{gwf_name_n}.hds"),
        ("GWFBUDGET", f"{rel_gwf_ws_n}/{gwf_name_n}.cbb"),
    ],
)
flopy.mf6.ModflowEms(prt_sim_n, pname="ems", filename=f"{prt_name_n}.ems")
prt_sim_n.write_simulation(silent=True)
success, buff = prt_sim_n.run_simulation(silent=False)
assert success, "MODFLOW 6 did not terminate normally"

#### Compare the two passes

Plot the two runs side by side over each model's own head field — Phase 1 (drape) on the left, Phase 3 (Newton) on the right.

In [ ]:
pls_prt_n = pd.read_csv(prt_ws_n / f"{prt_name_n}.csv")
h_p1 = gwf.output.head().get_data(kstpkper=(0, 1))[1]
h_p2 = gwf_n.output.head().get_data(kstpkper=(0, 1))[1]

with flopy.plot.styles.USGSPlot():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, model, pls, harr, title in [
        (axes[0], gwf, pls_prt, h_p1, "Phase 1: no Newton, drape"),
        (
            axes[1],
            gwf_n,
            pls_prt_n,
            h_p2,
            "Phase 3: Newton, dry_tracking_method='drop'",
        ),
    ]:
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=10)
        mm = flopy.plot.PlotMapView(model, ax=ax, layer=0)
        mm.plot_grid(alpha=0.2, lw=0.4)
        mm.plot_array(harr, alpha=0.25, vmin=1582, vmax=1587)
        mm.plot_bc("WEL", plotAll=True, kper=1, color="red")
        mm.plot_bc("CHD", plotAll=True, color="steelblue")
        mm.plot_pathline(pls, layer="all", alpha=0.5, linewidth=0.5)
        ax.scatter(pls.x, pls.y, c=pls.t, s=5)
    axes[1].legend(
        handles=[Patch(color="red", label="WELL"), Patch(color="blue", label="CHD")],
    )
    fig.tight_layout()
    plt.show()

**What to look for.** The two runs trace nearly the same lateral pathways to the constant-head outlet, but reach them through different branches of the decision tree. On the left the dry top cells are inactive, so `drape` drops each particle to the water table **at release**. On the right Newton keeps those cells active, so particles start inside them and `dry_tracking_method` drops them to the water table **during tracking**. Same physics, two mechanisms — and with Newton you could switch to `'stop'` or `'stay'` to change what happens to particles in dry cells.

## Recap

In this notebook you:

- Built a transient MODFLOW 6 **GWF** model whose upper layers dry out and re-wet, using convertible cells (NPF `icelltype=1`) and storage (STO).
- Saved the flow model's heads and cell budgets, and confirmed the top layer goes dry after the first stress period.
- Ran particle tracking two ways. **Without Newton** the dry cells are inactive, so PRT used `drape=True` to drop particles onto the water table at release. **With Newton** the dry cells stay active, so PRT tracked particles through them and `dry_tracking_method="drop"` dropped them to the water table during tracking.
- Plotted the pathlines in map view (colored by travel time), in 3D with PyVista, and side by side to compare the two passes.
- Walked the release and tracking decision trees that govern PRT's behavior in dry cells.